# PyMC-Marketing MMM: **DMA-level (panel)** or national (aggregated)

This notebook fits **PyMC-Marketing**'s multidimensional MMM on `examples/data/MMM Data.csv`.

**Policy:** **`examples/dashboard_rmse_optimized.py` is not edited** for this workflow. **Panel** arrays come from **`load_real_mmm_data`** in that file (loaded via `importlib` — same code path as the dashboard). Then **`UnifiedDataPipeline.temporal_split`**, **`SimpleGlobalScaler`**, and long `DataFrame`s for PyMC — **up to `mmm.fit`**. After that, modeling is PyMC-Marketing only.

**Panel mode (`AGGREGATE_TO_NATIONAL = False`):** **`load_real_mmm_data`** from **`dashboard_rmse_optimized.py`**, then **`UnifiedDataPipeline.temporal_split`** with **`examples/pymc_aligned_dcm_config.json`** and **`SimpleGlobalScaler`** on controls only (fit on train). **Media** and **target** stay **raw** in the long table; PyMC-Marketing max-abs scales inside `MMM`.

**National mode:** Legacy aggregation path (sum media / mean controls); uses the same **holdout ratio** from DeepCausalMMM config; controls get **sklearn `StandardScaler`**.

**Speed:** Use **`MAX_DMAS`** (e.g. 25–50) or **`MCMC_PROFILE = "smoke"`** for quick runs.

**Holdout:** Last fraction of calendar weeks per DMA (config — default **12%**). Use `predict(..., include_last_observations=True)` on holdout for adstock continuity.

**R² caveat:** Reported R² is **sklearn on full posterior mean μ vs y** (seasonality + controls + media, in-sample on train). That often looks **much higher** than “media-only” or vendor-style MMM fit stats — see the **Metrics** section below.

**Requirements:** Python 3.11+, `pip install pymc-marketing` (optional: `numpyro`). **`deepcausalmmm`** is loaded from **this repo** (adds repo root to `sys.path`); no `pip install deepcausalmmm` required. You still need **`torch`** (DeepCausalMMM dependency) for `UnifiedDataPipeline` / `SimpleGlobalScaler`. Edit **`examples/pymc_aligned_dcm_config.json`** to change holdout/scaling defaults.


## Configuration

In [20]:
import json
import os
import sys
from pathlib import Path


def _deepcausalmmm_repo_root() -> Path:
    """Find repo root (directory with pyproject.toml + deepcausalmmm package)."""
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / "pyproject.toml").is_file() and (d / "deepcausalmmm").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find deepcausalmmm repo root. "
        "cd to the repo root or examples/ (or open the project folder) and re-run."
    )


REPO_ROOT = _deepcausalmmm_repo_root()
_repo = str(REPO_ROOT)
if _repo not in sys.path:
    sys.path.insert(0, _repo)

# Aligned split + scaler settings (edit this file; mirrors get_default_config() subset)
_cfg_path = REPO_ROOT / "examples" / "pymc_aligned_dcm_config.json"
if not _cfg_path.is_file():
    raise FileNotFoundError(f"Missing {_cfg_path}")
with open(_cfg_path, encoding="utf-8") as f:
    DCM_CONFIG = json.load(f)

# Path to CSV (run notebook from repo root, or adjust)
DATA_PATH = os.path.join("examples", "data", "MMM Data.csv")
if not os.path.isfile(DATA_PATH):
    DATA_PATH = os.path.join("data", "MMM Data.csv")

# Panel: cap DMAs for runtime (None = all DMAs in CSV, matches dashboard).
MAX_DMAS = None

AGGREGATE_TO_NATIONAL = False

# MCMC profiles
MCMC_PROFILE = "smoke"
if MCMC_PROFILE == "smoke":
    CHAINS, DRAW, TUNE, TARGET_ACCEPT = 1, 150, 400, 0.90
elif MCMC_PROFILE == "fast":
    CHAINS, DRAW, TUNE, TARGET_ACCEPT = 2, 200, 800, 0.92
else:
    CHAINS, DRAW, TUNE, TARGET_ACCEPT = 4, 500, 2000, 0.95

_ho = DCM_CONFIG.get("holdout_ratio", 0.12)
print(
    f"REPO_ROOT={REPO_ROOT} | config={_cfg_path.name} | "
    f"MCMC profile={MCMC_PROFILE!r} | chains={CHAINS} draws={DRAW} tune={TUNE} | "
    f"aggregate_national={AGGREGATE_TO_NATIONAL} max_dmas={MAX_DMAS} | "
    f"holdout_ratio={_ho}"
)

RANDOM_SEED = 42
USE_NUMPYRO_NUTS = True


REPO_ROOT=/Users/adityapu/Documents/GitHub/deepcausalmmm | config=pymc_aligned_dcm_config.json | MCMC profile='smoke' | chains=1 draws=150 tune=400 | aggregate_national=False max_dmas=None | holdout_ratio=0.12


## Imports

In [21]:
import importlib.util
import sys
import time
import warnings

import numpy as np
import pandas as pd
import xarray as xr
from pymc_extras.prior import Prior
from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.multidimensional import MMM
from pymc_marketing.mmm.scaling import Scaling, VariableScaling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Local package from repo (works if Config cell ran; else resolve repo root here)
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "pyproject.toml").is_file() and (_d / "deepcausalmmm").is_dir():
        _s = str(_d)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
else:
    raise ImportError(
        "Could not find deepcausalmmm repo root. Run the Configuration cell first, "
        "or open the deepcausalmmm project folder and run from repo root or examples/."
    )

from deepcausalmmm.core.data import UnifiedDataPipeline
from deepcausalmmm.core.scaling import SimpleGlobalScaler

warnings.filterwarnings("ignore", category=UserWarning)


## Load and prepare data

**Panel:** Call **`dashboard_rmse_optimized.load_real_mmm_data`** (same implementation as the dashboard), then DCM **`temporal_split`** + **`SimpleGlobalScaler`** and assembly of **`X_train` / `X_holdout`**. Config: `examples/pymc_aligned_dcm_config.json`.

**National:** Legacy long-form path below.


In [22]:
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"MMM data not found: {DATA_PATH}")

geo_col = "dmacode"
HOLDOUT_RATIO = float(DCM_CONFIG.get("holdout_ratio", 0.12))

if AGGREGATE_TO_NATIONAL:
    target_col = "target_visits"
    # ----- Legacy national series (not dashboard-aligned on aggregation) -----
    df_raw = pd.read_csv(DATA_PATH)
    impression_cols = [c for c in df_raw.columns if c.startswith("impressions_")]
    control_cols = [c for c in df_raw.columns if c.startswith("control_")]
    region_col = "dmacode"
    time_col = "weekid"

    if MAX_DMAS is not None:
        keep = sorted(df_raw[region_col].unique())[: int(MAX_DMAS)]
        df_raw = df_raw[df_raw[region_col].isin(keep)].copy()

    n_channels = len(impression_cols)
    channel_short = [f"ch{i:02d}" for i in range(1, n_channels + 1)]
    df = df_raw.rename(columns=dict(zip(impression_cols, channel_short)))
    df = df.sort_values([region_col, time_col])

    for c in channel_short:
        df[c] = df[c].fillna(0)
    for c in control_cols + [target_col]:
        df[c] = df.groupby(region_col)[c].ffill()

    weeks_sorted = sorted(df[time_col].unique())
    week_to_date = {
        w: pd.Timestamp("2020-01-06") + pd.Timedelta(weeks=i) for i, w in enumerate(weeks_sorted)
    }
    df["date"] = df[time_col].map(week_to_date)
    _cal_sparse = sorted(df["date"].dropna().unique())
    _ncs = len(_cal_sparse)
    _twc = _ncs - int(_ncs * HOLDOUT_RATIO)
    _cut_sparse = _cal_sparse[_twc - 1]
    _mask_train_dates = df["date"] <= _cut_sparse
    for c in control_cols + [target_col]:
        if df[c].isna().any():
            _med = df.loc[_mask_train_dates, c].median()
            if pd.isna(_med):
                _med = float(df.loc[_mask_train_dates, c].mean())
            df[c] = df[c].fillna(_med)

    agg_map = {c: "sum" for c in channel_short + [target_col]}
    agg_map.update({c: "mean" for c in control_cols})
    df = df.groupby("date", as_index=False).agg(agg_map).sort_values("date").reset_index(drop=True)
    geo_col = "geo"
    df[geo_col] = "national"

    feature_cols = ["date", geo_col] + channel_short + control_cols
    df = df[feature_cols + [target_col]].sort_values([geo_col, "date"]).reset_index(drop=True)
    week_order = sorted(df["date"].unique())
    df["week_idx"] = df["date"].map({d: i for i, d in enumerate(week_order)})
    print(f"National series | Weeks: {len(week_order)} | Channels: {n_channels}")
    df.head()

else:
    # Panel: same arrays as dashboard — examples/dashboard_rmse_optimized.load_real_mmm_data
    from pathlib import Path

    _dash_py = REPO_ROOT / "examples" / "dashboard_rmse_optimized.py"
    if not _dash_py.is_file():
        raise FileNotFoundError(f"Missing {_dash_py}")
    _spec = importlib.util.spec_from_file_location("dashboard_rmse_optimized", _dash_py)
    _dash = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_dash)

    _data_abs = Path(DATA_PATH)
    if not _data_abs.is_file():
        _data_abs = REPO_ROOT / DATA_PATH
    if not _data_abs.is_file():
        _data_abs = REPO_ROOT / "data" / "MMM Data.csv"
    if not _data_abs.is_file():
        raise FileNotFoundError(f"MMM data not found: {DATA_PATH}")

    X_media, X_control, y, media_names, control_names = _dash.load_real_mmm_data(str(_data_abs))

    hdr = pd.read_csv(_data_abs, nrows=0)
    impression_cols = [c for c in hdr.columns if "impressions_" in c]
    if "target_visits" in hdr.columns:
        target_col = "target_visits"
        value_cols = [c for c in hdr.columns if c.startswith("control_")]
    else:
        target_col = "value_visits_visits"
        value_cols = [c for c in hdr.columns if "value_" in c and c != "value_visits_visits"]

    regions = sorted(pd.read_csv(_data_abs, usecols=["dmacode"])["dmacode"].unique().tolist())
    if MAX_DMAS is not None:
        regions = regions[: int(MAX_DMAS)]
        _k = len(regions)
        X_media, X_control, y = X_media[:_k], X_control[:_k], y[:_k]

    n_weeks = int(X_media.shape[1])

    pipeline = UnifiedDataPipeline(DCM_CONFIG)
    train_data, holdout_data = pipeline.temporal_split(X_media, X_control, y)
    train_weeks = int(train_data["X_media"].shape[1])
    holdout_weeks = int(holdout_data["X_media"].shape[1])
    assert train_weeks + holdout_weeks == n_weeks

    scaler = SimpleGlobalScaler(DCM_CONFIG)
    scaler.fit(train_data["X_media"], train_data["X_control"], train_data["y"])
    _, X_ctrl_train_t, _ = scaler.transform(
        train_data["X_media"], train_data["X_control"], train_data["y"]
    )
    _, X_ctrl_hold_t, _ = scaler.transform(
        holdout_data["X_media"], holdout_data["X_control"], holdout_data["y"]
    )
    X_ctrl_train = X_ctrl_train_t.numpy()
    X_ctrl_hold = X_ctrl_hold_t.numpy()

    channel_short = [f"ch{i:02d}" for i in range(1, len(impression_cols) + 1)]
    control_cols = list(value_cols)

    dates_all = [pd.Timestamp("2020-01-06") + pd.Timedelta(weeks=i) for i in range(n_weeks)]

    rows_tr = []
    for ri, code in enumerate(regions):
        for ti in range(train_weeks):
            row = {geo_col: code, "date": dates_all[ti]}
            for c in range(len(channel_short)):
                row[channel_short[c]] = float(train_data["X_media"][ri, ti, c])
            for c in range(len(control_cols)):
                row[control_cols[c]] = float(X_ctrl_train[ri, ti, c])
            row[target_col] = float(train_data["y"][ri, ti])
            rows_tr.append(row)
    df_train = pd.DataFrame(rows_tr).sort_values([geo_col, "date"]).reset_index(drop=True)

    rows_ho = []
    for ri, code in enumerate(regions):
        for ti in range(holdout_weeks):
            gti = train_weeks + ti
            row = {geo_col: code, "date": dates_all[gti]}
            for c in range(len(channel_short)):
                row[channel_short[c]] = float(holdout_data["X_media"][ri, ti, c])
            for c in range(len(control_cols)):
                row[control_cols[c]] = float(X_ctrl_hold[ri, ti, c])
            row[target_col] = float(holdout_data["y"][ri, ti])
            rows_ho.append(row)
    df_holdout = pd.DataFrame(rows_ho).sort_values([geo_col, "date"]).reset_index(drop=True)

    X_train = df_train.drop(columns=[target_col])
    y_train = df_train[target_col].copy()
    y_train.name = target_col
    X_holdout = df_holdout.drop(columns=[target_col])
    y_holdout = df_holdout[target_col].copy()

    week_order = dates_all
    n_channels = len(channel_short)

    print(
        f"Panel (dashboard_rmse_optimized.load_real_mmm_data) | DMAs: {len(regions)} | "
        f"weeks train/hold: {train_weeks}/{holdout_weeks} | holdout_ratio={HOLDOUT_RATIO} | channels: {n_channels}"
    )
    df_train.head()



INFO - 
 Unified Data Pipeline - Time Series Split (Ratio-Based):
INFO -     Training: 96 actual weeks (weeks 1-96) - 88.1%
INFO -     Holdout: 13 actual weeks (weeks 97-109) - 11.9%
INFO -     Burn-in Padding: 6 weeks will be added to BOTH train and holdout
INFO -     Model sees: Train 102 weeks, Holdout 19 weeks
INFO -     Evaluation: Remove 6 padding weeks, evaluate on ALL actual data
INFO -     Time series approach: Training on historical data, testing on most recent data
Panel (load_real_mmm_data logic, inlined) | DMAs: 190 | weeks train/hold: 96/13 | holdout_ratio=0.12 | channels: 13


## Train / holdout split and control scaling (national mode only; panel is done above)


In [23]:
if AGGREGATE_TO_NATIONAL:
    n_weeks = len(week_order)
    train_weeks = n_weeks - int(n_weeks * HOLDOUT_RATIO)
    holdout_weeks = n_weeks - train_weeks

    df["is_holdout"] = df["week_idx"] >= train_weeks
    df_train = df[~df["is_holdout"]].drop(columns=["is_holdout", "week_idx"])
    df_holdout = df[df["is_holdout"]].drop(columns=["is_holdout", "week_idx"])

    X_train = df_train.drop(columns=[target_col])
    y_train = df_train[target_col].copy()
    y_train.name = target_col
    X_holdout = df_holdout.drop(columns=[target_col])
    y_holdout = df_holdout[target_col].copy()

    ctrl_scaler = StandardScaler()
    X_train[control_cols] = ctrl_scaler.fit_transform(X_train[control_cols])
    X_holdout[control_cols] = ctrl_scaler.transform(X_holdout[control_cols])
else:
    pass  # train/holdout already built; controls already DCM-scaled

print(
    f"Train: {len(X_train)} rows ({train_weeks} weeks) | Holdout: {len(X_holdout)} rows ({holdout_weeks} weeks)"
)


Train: 18240 rows (96 weeks) | Holdout: 2470 rows (13 weeks)


## Build pooled MMM and sample

In [24]:
adstock = GeometricAdstock(
    l_max=8,
    priors={"alpha": Prior("Beta", alpha=2, beta=5, dims="channel")},
)
saturation = LogisticSaturation(
    priors={
        "lam": Prior("Gamma", alpha=3, beta=1, dims="channel"),
        "beta": Prior("HalfNormal", sigma=2, dims="channel"),
    },
)
pool_model_config = {
    "gamma_control": Prior("Normal", mu=0, sigma=2, dims="control"),
}
scaling = Scaling(
    target=VariableScaling(method="max", dims=()),
    channel=VariableScaling(method="max", dims=()),
)

mmm = MMM(
    date_column="date",
    target_column=target_col,
    channel_columns=channel_short,
    control_columns=control_cols,
    dims=(geo_col,),
    adstock=adstock,
    saturation=saturation,
    yearly_seasonality=2,
    scaling=scaling,
    model_config=pool_model_config,
)

fit_kw = dict(
    chains=CHAINS,
    draws=DRAW,
    tune=TUNE,
    target_accept=TARGET_ACCEPT,
    random_seed=RANDOM_SEED,
    progressbar=True,
)
if USE_NUMPYRO_NUTS:
    try:
        import numpyro  # noqa: F401

        fit_kw["nuts_sampler"] = "numpyro"
    except ImportError:
        print("numpyro not installed; using default PyMC NUTS")

t0 = time.perf_counter()
mmm.fit(X_train, y_train, **fit_kw)
fit_s = time.perf_counter() - t0
print(f"Fit wall time: {fit_s:.1f} s")

numpyro not installed; using default PyMC NUTS


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [intercept_contribution, adstock_alpha, saturation_lam, saturation_beta, gamma_control, gamma_fourier, y_sigma]


Sampling 1 chain for 400 tune and 150 draw iterations (400 + 150 draws total) took 5871 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks


Fit wall time: 5873.5 s


## Metrics: training and holdout

**DeepCausalMMM `ModelTrainer`** (`deepcausalmmm/core/trainer.py`): `final_train_r2` / `final_holdout_r2` and RMSE use `sklearn.metrics.r2_score` / `mean_squared_error` on **flattened** arrays after `SimpleGlobalScaler.inverse_transform_target` — i.e. **original-scale** `y` vs model predictions.

**This notebook (PyMC):** same protocol — **original-scale** `y_train` / `y_holdout` vs **posterior mean** `μ`, with PyMC’s target max-abs scaling undone via `mmm.scalers._target` (global or per-geo, matching the fitted `MMM`). **Panel:** `predict` is aligned to each `(date, dmacode)` row via `_align_panel_mu` before scoring (avoids C-order vs `(geo, date)` sort mismatch).

Not LOO / not full posterior-predictive draws of `y` — mean structure only, like comparing to the neural net’s point prediction in DCM.

**Why training R² can look “too high” vs typical MMM dashboards:** `sklearn.metrics.r2_score` here scores the **entire** fitted mean — **intercept + yearly Fourier seasonality + controls + adstocked media** (and pooling across geos). That is **not** the same as “R² from media alone” or Robyn-style decomp summaries. With **many panel rows** and **strong predictable seasonality/levels**, **in-sample** train R² often lands **very high** even when **incremental media** is still uncertain. Treat **`holdout_r2`** (and **`holdout_rmse`** vs **`holdout_rmse_naive_train_mean`**) as the more decision-relevant checks; do **not** compare these absolute R² numbers to another tool unless the **same definition** and **same split** are used.


In [25]:
# predict() returns mu in *target-scaled* model space on a (date × geo) grid in C-order
# (date index slow, geo fast), which is NOT the same row order as X_train when X is sorted by (geo, date).
# Merge on (date, geo) so each row's μ pairs with the correct y before R².
def _align_panel_mu(mmm, X_df, pred, geo_col, date_col="date"):
    pred = np.asarray(pred, dtype=float).ravel()
    dates = pd.to_datetime(pd.Index(sorted(X_df[date_col].unique())))
    geos = np.asarray(mmm.model_coords[geo_col])
    n_d, n_g = len(dates), len(geos)
    if pred.size != n_d * n_g:
        raise ValueError(
            f"predict length {pred.size} != {n_d} dates × {n_g} geos = {n_d * n_g}. "
            "Check X dates/geos vs model_coords."
        )
    wide = pred.reshape(n_d, n_g, order="C")
    mi = pd.MultiIndex.from_product([dates, geos], names=[date_col, geo_col])
    long = pd.DataFrame({"_mu_scaled": wide.ravel(order="C")}, index=mi).reset_index()
    left = X_df[[date_col, geo_col]].copy()
    left[date_col] = pd.to_datetime(left[date_col])
    # Avoid silent merge misses (int dmacode vs str in coords)
    long[geo_col] = long[geo_col].astype(left[geo_col].dtype)
    long[date_col] = pd.to_datetime(long[date_col])
    out = left.merge(long, on=[date_col, geo_col], how="left")
    if out["_mu_scaled"].isna().any():
        miss = int(out["_mu_scaled"].isna().sum())
        raise ValueError(
            f"Missing μ for {miss} rows after (date, geo) merge — geo/date dtypes must match model_coords."
        )
    return out["_mu_scaled"].to_numpy(dtype=float)


# Model likelihood is on max-abs scaled y; undo with global (or per-geo) target scale from fit.
y_train_pred = mmm.predict(
    X_train, extend_idata=False, include_last_observations=False
)
y_holdout_pred = mmm.predict(
    X_holdout, extend_idata=False, include_last_observations=True
)

if not AGGREGATE_TO_NATIONAL:
    y_train_pred = _align_panel_mu(mmm, X_train, y_train_pred, geo_col)
    y_holdout_pred = _align_panel_mu(mmm, X_holdout, y_holdout_pred, geo_col)
else:
    y_train_pred = np.asarray(y_train_pred, dtype=float).ravel()
    y_holdout_pred = np.asarray(y_holdout_pred, dtype=float).ravel()

ts = mmm.scalers._target
if len(ts.dims) == 0:
    _fac = float(np.asarray(ts).reshape(-1)[0])
    y_train_pred = y_train_pred * _fac
    y_holdout_pred = y_holdout_pred * _fac
else:
    _dn = list(ts.dims)[0]
    _scale_ser = pd.Series(
        np.asarray(ts.values).ravel(),
        index=pd.Index(ts.coords[_dn].values, name=_dn),
    )
    _ftr = X_train[geo_col].map(_scale_ser).astype(float).to_numpy()
    _fho = X_holdout[geo_col].map(_scale_ser).astype(float).to_numpy()
    y_train_pred = y_train_pred * _ftr
    y_holdout_pred = y_holdout_pred * _fho

# Match ModelTrainer: flattened float64 vectors, original-scale y (trainer: .detach().cpu().numpy().ravel())
y_true_tr = np.asarray(y_train, dtype=np.float64).ravel()
y_true_ho = np.asarray(y_holdout, dtype=np.float64).ravel()
y_train_pred = np.asarray(y_train_pred, dtype=np.float64).ravel()
y_holdout_pred = np.asarray(y_holdout_pred, dtype=np.float64).ravel()
assert y_true_tr.shape == y_train_pred.shape, (y_true_tr.shape, y_train_pred.shape)
assert y_true_ho.shape == y_holdout_pred.shape, (y_true_ho.shape, y_holdout_pred.shape)
assert np.isfinite(y_train_pred).all() and np.isfinite(y_holdout_pred).all()

train_r2 = r2_score(y_true_tr, y_train_pred)
train_rmse = float(np.sqrt(mean_squared_error(y_true_tr, y_train_pred)))

holdout_r2 = r2_score(y_true_ho, y_holdout_pred)
holdout_rmse = float(np.sqrt(mean_squared_error(y_true_ho, y_holdout_pred)))

# Cross-check: manual R² = 1 - SSE/SST (must match sklearn up to float noise)
_sst = np.sum((y_true_tr - y_true_tr.mean()) ** 2)
_sse = np.sum((y_true_tr - y_train_pred) ** 2)
_r2_manual = 1.0 - _sse / _sst if _sst > 0 else float("nan")
if not np.isfinite(train_r2) or abs(train_r2 - _r2_manual) > 1e-5:
    raise AssertionError(f"train R² mismatch sklearn vs manual: {train_r2} vs {_r2_manual}")

_naive = np.full(shape=y_true_ho.shape[0], fill_value=float(np.mean(y_true_tr)))
holdout_rmse_naive_train_mean = float(
    np.sqrt(mean_squared_error(y_true_ho, _naive))
)

_n_tr = int(len(y_train))
_n_ho = int(len(y_holdout))

out = pd.DataFrame(
    {
        "metric": [
            "aggregate_to_national",
            "holdout_ratio",
            "n_train_rows",
            "n_holdout_rows",
            "training_r2",
            "holdout_r2",
            "training_rmse",
            "holdout_rmse",
            "holdout_rmse_naive_train_mean",
            "fit_runtime_s",
        ],
        "value": [
            float(AGGREGATE_TO_NATIONAL),
            HOLDOUT_RATIO,
            _n_tr,
            _n_ho,
            train_r2,
            holdout_r2,
            train_rmse,
            holdout_rmse,
            holdout_rmse_naive_train_mean,
            fit_s,
        ],
    }
)
try:
    display(out)
except NameError:
    print(out.to_string(index=False))

print(
    "Note: R² is sklearn on **original-scale y vs posterior mean μ** (not LOO, not posterior-predictive y). "
    "Train R² is in-sample for that μ and is often high with flexible mean + many panel rows."
)


Sampling: [y]


Sampling: [y]


,metric,value
0,aggregate_to_national,0.000000e+00
1,holdout_ratio,1.200000e-01
2,n_train_rows,1.824000e+04
3,n_holdout_rows,2.470000e+03
4,training_r2,9.937351e-01
5,holdout_r2,9.025755e-01
6,training_rmse,1.094351e+05
7,holdout_rmse,4.808256e+05
8,holdout_rmse_naive_train_mean,1.544695e+06
9,fit_runtime_s,5.873536e+03


Note: R² is sklearn on **original-scale y vs posterior mean μ** (not LOO, not posterior-predictive y). Train R² is in-sample for that μ and is often high with flexible mean + many panel rows.
